In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import time
import unyt as u

import richio
import dev

## Nozzle check

Back-of-envelope check against [arXiv:2510.04790](https://arxiv.org/abs/2510.04790), which finds
the nozzle-shock dissipation to be ~4e-5 of the orbital energy (their order-of-magnitude estimate
is `dE/E ~ (v_th/v_bulk)^2 ~ 1e-4`, converging to ~4e-5 at their highest resolution). Their actual
measured quantity is the **ratio of thermal (internal) energy to kinetic energy** near pericenter.

Our sim (`R0.47M0.5BH10000beta1S60n1.5ComptonHiResNewAMR`) is a full (beta=1) disruption too, but
Mbh=1e4 Msun instead of their 1e6 Msun.

Matching their setup:

- **Frame.** The first ~21 snapshots are in the star's comoving frame; from snap_22 on the sim is
  in the **global BH frame**, so the snapshot velocity field is the true orbital velocity. We only
  measure in the BH frame.
- **Time.** One fallback time is `t_fb = 2.577726 day`, so `t/t_fb = tfb[day] / 2.577726`. We
  evaluate around **0.3 t_fb**.
- **Selection.** The default is the pericenter/nozzle region used in the comparison calculation:
  center the angular cut on the maximum-dissipation cell, keep +/- 4.5 deg, and require
  `r < 1100 Rs`. Set `selection_mode = "peak_kinetic_beam"` below to recover the old
  distant-stream selection for comparison.

We compute two things in that selection:

1. **Internal/kinetic** `sum(IE*mass) / sum(KE)` -> accumulated thermal energy relative to
   kinetic energy in exactly the same selected cells.
2. **Rate construction** `(sum(Ediss_rate*Volume) / sum(KE)) * (L / v_peri)` -> the instantaneous
   dissipation *rate* turned into an energy over one pericenter crossing, with `L = Rp` and
   `v_peri = sqrt(2 G Mbh / Rp)`.

In [2]:
# --------------------------- CHANGE SETTINGS HERE ---------------------------
base = "/data1/projects/pi-rossiem/TDE_data/R0.47M0.5BH10000beta1S60n1.5ComptonHiResNewAMR"
snapshot_number = 45
time_scan_snapshots = [*range(22, 45, 2), 45, *range(46, 54, 2)]
selection_mode = "max_dissipation_nozzle"  # or "peak_kinetic_beam"
star_only = False                     # False reproduces the screenshot; True applies mask_star_ratio()
half_width_deg = 4.5                 # max-dissipation nozzle: +/- this angle
radial_limit_rs = 1100.0             # max-dissipation nozzle: r < this many Rs
beam_width_deg = 3.0                 # old peak-KE method
neighboring_beams = 1                # old peak-KE method: peak +/- this many beams
# ---------------------------------------------------------------------------

mstar = 0.5 * richio.units.mscale
rstar = 0.47 * richio.units.lscale
mbh = 1e4 * richio.units.mscale
G = u.physical_constants.G
t_fb = 2.577726 * u.day  # fallback time for this system (richio NPY tfb unit)

rt = rstar * (mbh / mstar) ** (1 / 3)
rp = rt  # beta = 1, full disruption
v_peri = np.sqrt(2 * G * mbh / rp)  # global BH-frame speed at pericenter
schwarzschild_radius = 2 * G * mbh / u.physical_constants.clight**2

peri_length = rp  # "length of pericenter passage" ~ Rp
crossing_time = peri_length / v_peri  # let unyt keep the units; convert only for display

print(f"rp = {rp.to('cm'):.3e}")
print(f"v_peri = {v_peri.to('km/s'):.3e}")
print(f"peri_length = Rp = {peri_length.to('cm'):.3e}, crossing_time = {crossing_time.to('s'):.3e}")
print(f"t_fb = {t_fb}, 0.3 t_fb = {0.3 * t_fb}")

rp = 8.930e+11 cm
v_peri = 1.729e+04 km/s
peri_length = Rp = 8.930e+11 cm, crossing_time = 5.165e+02 s
t_fb = 2.577726 day, 0.3 t_fb = 0.7733178 day


## BH-frame times

Scan `t/t_fb` across BH-frame snapshots (snap_22 on). The diagnostic below uses the visible
`snapshot_number` setting; it defaults to snap_45, the snapshot in the comparison screenshot.

In [3]:
for i in time_scan_snapshots:
    snap_i = richio.load(f"{base}/snap_{i}")
    t_day = snap_i.tfb.in_units("day")
    print(f"snap_{i:3d}  t = {float(t_day):.4f} d   t/t_fb = {float(t_day / t_fb):.4f}")

snap_ 22  t = 0.1379 d   t/t_fb = 0.0535
snap_ 24  t = 0.1709 d   t/t_fb = 0.0663
snap_ 26  t = 0.2083 d   t/t_fb = 0.0808
snap_ 28  t = 0.2503 d   t/t_fb = 0.0971
snap_ 30  t = 0.2972 d   t/t_fb = 0.1153
snap_ 32  t = 0.3503 d   t/t_fb = 0.1359
snap_ 34  t = 0.4078 d   t/t_fb = 0.1582
snap_ 36  t = 0.4711 d   t/t_fb = 0.1828
snap_ 38  t = 0.5408 d   t/t_fb = 0.2098
snap_ 40  t = 0.6163 d   t/t_fb = 0.2391
snap_ 42  t = 0.6981 d   t/t_fb = 0.2708
snap_ 44  t = 0.7867 d   t/t_fb = 0.3052
snap_ 45  t = 0.8336 d   t/t_fb = 0.3234
snap_ 46  t = 0.8822 d   t/t_fb = 0.3422
snap_ 48  t = 0.9863 d   t/t_fb = 0.3826
snap_ 50  t = 1.0966 d   t/t_fb = 0.4254
snap_ 52  t = 1.2147 d   t/t_fb = 0.4712


## Dissipation fraction

Every loaded field, mask, intermediate sum, and final ratio is exposed directly in the cells
below; there are no helper functions. The default, `max_dissipation_nozzle`,
matches the screenshot: center on the maximum-dissipation cell, apply the angular and radial
cuts, and evaluate the local nozzle. `peak_kinetic_beam` preserves the old method as an easy
switch, but it selects the distant stream in this snapshot.

Two rate fractions are printed. The **summed-KE** value uses the kinetic energy of every selected
cell. The **local-velocity proxy** reproduces the screenshot's approximation using the velocity
of the maximum-dissipation cell.

In [ ]:
# Load the selected snapshot and expose every field used below.
snap = richio.load(f"{base}/snap_{snapshot_number}")
field_mask = snap.mask_star_ratio() if star_only else np.ones(len(snap.density), dtype=bool)

x = snap.CMx[field_mask]
y = snap.CMy[field_mask]
z = snap.CMz[field_mask]
vx = snap.velocity_x[field_mask]
vy = snap.velocity_y[field_mask]
vz = snap.velocity_z[field_mask]
volume = snap.volume[field_mask]
density = snap.density[field_mask]
specific_internal_energy = snap.internal_energy[field_mask]
dissipation_density = snap.dissipation[field_mask]

cell_mass = density * volume
speed_squared = vx**2 + vy**2 + vz**2
kinetic_energy = 0.5 * cell_mass * speed_squared
thermal_energy = specific_internal_energy * cell_mass
dissipation_power = dissipation_density * volume

radius = np.sqrt(x**2 + y**2 + z**2)
azimuth = np.arctan2(np.asarray(y), np.asarray(x))
t_over_tfb = float(snap.tfb / t_fb)

In [ ]:
# Construct both selections explicitly so every intermediate array is inspectable.

# 1. Local pericenter/nozzle region centered on maximum dissipation.
maximum_dissipation_index = int(np.argmax(dissipation_density))
maximum_dissipation_longitude = azimuth[maximum_dissipation_index]
maximum_radius = radial_limit_rs * schwarzschild_radius

nozzle_angular_distance = (
    azimuth - maximum_dissipation_longitude + np.pi
) % (2 * np.pi) - np.pi
nozzle_angular_mask = np.abs(nozzle_angular_distance) < np.deg2rad(half_width_deg)
nozzle_radial_mask = radius < maximum_radius
max_dissipation_nozzle_selection = nozzle_radial_mask & nozzle_angular_mask

# 2. Old global peak-kinetic-energy beam, retained as a switchable comparison.
number_of_beams = int(round(360.0 / beam_width_deg))
if not np.isclose(number_of_beams * beam_width_deg, 360.0):
    raise ValueError("beam_width_deg must divide 360 degrees")

beam_index = (
    np.floor((azimuth + np.pi) / (2 * np.pi) * number_of_beams).astype(int)
) % number_of_beams
kinetic_energy_per_beam = np.bincount(
    beam_index,
    weights=np.asarray(kinetic_energy),
    minlength=number_of_beams,
)
peak_kinetic_beam_index = int(np.argmax(kinetic_energy_per_beam))
selected_peak_beams = [
    (peak_kinetic_beam_index + offset) % number_of_beams
    for offset in range(-neighboring_beams, neighboring_beams + 1)
]
peak_kinetic_beam_selection = np.isin(beam_index, selected_peak_beams)
peak_kinetic_beam_longitude = (
    -np.pi + (peak_kinetic_beam_index + 0.5) * 2 * np.pi / number_of_beams
)

# The only switch: choose which already-visible mask feeds the sums below.
if selection_mode == "max_dissipation_nozzle":
    selection = max_dissipation_nozzle_selection
    selection_center = maximum_dissipation_longitude
elif selection_mode == "peak_kinetic_beam":
    selection = peak_kinetic_beam_selection
    selection_center = peak_kinetic_beam_longitude
else:
    raise ValueError(f"Unknown selection_mode: {selection_mode!r}")

In [ ]:
# Sum the selected cells. These are the complete intermediate quantities.
selected_cell_count = int(np.count_nonzero(selection))
selected_mass = np.sum(cell_mass[selection])
selected_kinetic_energy = np.sum(kinetic_energy[selection])
selected_thermal_energy = np.sum(thermal_energy[selection])
selected_dissipation_power = np.sum(dissipation_power[selection])

# Screenshot approximation: use the speed of the one maximum-dissipation cell.
local_speed_squared = speed_squared[maximum_dissipation_index]
local_specific_kinetic_energy = 0.5 * local_speed_squared
deposited_specific_energy = (
    selected_dissipation_power * crossing_time / selected_mass
)

# Final dimensionless ratios.
internal_over_kinetic = (
    selected_thermal_energy / selected_kinetic_energy
).to("dimensionless")
rate_fraction_summed_kinetic = (
    selected_dissipation_power * crossing_time / selected_kinetic_energy
).to("dimensionless")
rate_fraction_local_velocity = (
    deposited_specific_energy / local_specific_kinetic_energy
).to("dimensionless")

In [ ]:
print(f"snap_{snapshot_number}, t/t_fb = {t_over_tfb:.3f}")
print(f"selection_mode = {selection_mode!r}")
print(f"star_only = {star_only}")
print(f"maximum-dissipation longitude = {np.rad2deg(maximum_dissipation_longitude):.2f} deg")
print(f"peak-KE beam longitude = {np.rad2deg(peak_kinetic_beam_longitude):.2f} deg")
print(f"active selection center = {np.rad2deg(selection_center):.2f} deg")
print(f"radial limit = {radial_limit_rs:g} Rs")
print(f"crossing time = {crossing_time.to('s'):.1f}")
print()
print(f"selected cells                      = {selected_cell_count:,}")
print(f"selected mass                       = {selected_mass.to('g'):.3e}")
print(f"dissipation power                   = {selected_dissipation_power.to('erg/s'):.3e}")
print(f"summed kinetic energy               = {selected_kinetic_energy.to('erg'):.3e}")
print(f"summed thermal energy               = {selected_thermal_energy.to('erg'):.3e}")
print(f"local specific kinetic energy       = {local_specific_kinetic_energy.to('erg/g'):.3e}")
print(f"deposited specific energy           = {deposited_specific_energy.to('erg/g'):.3e}")
print()
print(f"internal/kinetic                    = {float(internal_over_kinetic):.6e}")
print(f"rate fraction, summed KE            = {float(rate_fraction_summed_kinetic):.6e}")
print(f"rate fraction, local-velocity proxy = {float(rate_fraction_local_velocity):.6e}")

## Switching and interpretation

Only edit the settings cell near the top:

- `snapshot_number = 45` reproduces the snapshot in the screenshot.
- `selection_mode = "max_dissipation_nozzle"` selects the local pericenter nozzle and is the default.
- `selection_mode = "peak_kinetic_beam"` restores the old global peak-KE selection. In snap_45 its
  center is about -157.5 deg, opposite the maximum-dissipation region at about -3.2 deg.
- `half_width_deg`, `radial_limit_rs`, `beam_width_deg`, `neighboring_beams`, and `star_only` are all
  exposed in the same settings cell.

For the default snap_45 nozzle selection, the local-velocity proxy should reproduce approximately
`0.005956`, while normalization by the actual summed kinetic energy gives approximately `0.006635`.
The summed-KE value is the dimensionally complete region-integrated diagnostic; the local-velocity
value is retained explicitly so the colleague's calculation can be reproduced exactly.